# Лекция 5. Полнотекстовый поиск: демонстрация

Курс «От интеллектуального поиска к интеллектуальным агентам», ВМК МГУ. О. М. Атаева, А. П. Халов.

Блокнот продолжает задачу редактора. Положение о рецензировании (31 запись, `data/corpus_Aprime_standards.csv`)
индексируется, запросы редактора ранжируются тремя способами, качество измеряется на размеченной выборке
(`data/queries_Aprime.csv`). Весь код поиска лежит в `search_bm25.py` и написан явно, без библиотек поиска.

| Шаг | Что делаем |
|---|---|
| 0 | Корпуса: положение и планиметрия |
| 1 | Конвейер индексации: токены, стоп-слова, основы, размер словаря |
| 2 | Инвертированный индекс и булев поиск |
| 3 | TF-IDF и косинус: выдача по вопросу редактора |
| 4 | BM25: та же выдача, разбор оценки, параметры k1 и b, мини-пример вручную |
| 5 | BM25F: поле названия с весом 2 |
| 6 | Тестовая выборка: 16 запросов с градациями |
| 7 | Метрики: точность@5, полнота@5, MRR, MAP, NDCG@10 для четырёх вариантов |
| 8 | NDCG@3 вручную и сверка с функцией |
| 9 | Три провала: синонимия, перефразировка, формула |
| 10 | Агент: инструмент поиска с измеренным качеством |
| 11 | Вариант по математике |

## Шаг 0. Корпуса

Положение о рецензировании из лекции 1: разделы, термины, требования. У каждой записи два текстовых поля,
название и формулировка. Рядом корпус планиметрии для варианта по математике.

In [1]:
import csv, math, collections, json
import pandas as pd
from search_bm25 import Pipeline, Index, STOP, tokenize, load_qrels, evaluate, \
    precision_at_k, recall_at_k, reciprocal_rank, average_precision, dcg, ndcg_at_k

pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 70)

def read_corpus(path):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    docs = [{"название": r["название"], "формулировка": r["формулировка"]} for r in rows]
    return rows, docs, [r["id"] for r in rows]

rows, docs, ids = read_corpus("data/corpus_Aprime_standards.csv")
rowsA, docsA, idsA = read_corpus("data/corpus_A_planimetry.csv")
text = {r["id"]: r["название"] + ". " + r["формулировка"] for r in rows + rowsA}
print("положение:", len(docs), "записей;", "планиметрия:", len(docsA), "утверждений")
pd.DataFrame(rows)[["id", "тип", "название", "формулировка"]].iloc[[0, 8, 19, 25]]

положение: 31 записей; планиметрия: 30 утверждений
     id         тип                          название                                                                                                            формулировка
0   S01      Раздел      Раздел 1. Область применения  Положение устанавливает требования к подготовке рецензированию и принятию решений по статьям представляемым в сборник.
8   T06      Термин                          Рецензия                                     Рецензия — письменное заключение рецензента о статье содержащее оценку и замечания.
19  S06      Раздел  Раздел 6. Порядок рецензирования                       Раздел устанавливает порядок назначения рецензентов содержание рецензии и порядок ответа авторов.
25  R66  Требование           6.6 Срок рецензирования                                               Рецензия представляется в течение 21 дня с момента назначения рецензента.


## Шаг 1. Конвейер индексации

Документ превращается в список токенов, из него убираются стоп-слова, оставшиеся слова приводятся к основе.
Те же шаги применяются к запросу. Каждый шаг меняет словарь, и размер словаря показывает, сколько словоформ
склеилось в одно слово индекса.

In [2]:
s = text["R66"]
print("текст:     ", s)
print("токены:    ", tokenize(s))
print("без стоп:  ", Pipeline(stop=True, stem=False)(s))
print("основы:    ", Pipeline(stop=True, stem=True)(s))
print()
print("одна основа у разных словоформ:", Pipeline()("рецензия рецензии рецензий рецензент рецензирования"))
print("стемминг склеивает разное:     ", Pipeline()("статья статьи стать"), " лемматизация различила бы «статья» и «стать»")
print("стемминг не склеивает близкое: ", Pipeline()("принятие принимает"), " это ещё отзовётся на шаге 4")

текст:      6.6 Срок рецензирования. Рецензия представляется в течение 21 дня с момента назначения рецензента.
токены:     ['6.6', 'срок', 'рецензирования', 'рецензия', 'представляется', 'в', 'течение', '21', 'дня', 'с', 'момента', 'назначения', 'рецензента']
без стоп:   ['6.6', 'срок', 'рецензирования', 'рецензия', 'представляется', 'течение', '21', 'дня', 'момента', 'назначения', 'рецензента']
основы:     ['6.6', 'срок', 'рецензирован', 'реценз', 'представля', 'течен', '21', 'дня', 'момент', 'назначен', 'рецензент']

одна основа у разных словоформ: ['реценз', 'реценз', 'реценз', 'рецензент', 'рецензирован']
стемминг склеивает разное:      ['стат', 'стат', 'стат']  лемматизация различила бы «статья» и «стать»
стемминг не склеивает близкое:  ['принят', 'принима']  это ещё отзовётся на шаге 4


In [3]:
pipes = {"без нормализации": Pipeline(stop=False, stem=False),
         "стоп-слова": Pipeline(stop=True, stem=False),
         "стоп-слова и основы": Pipeline(stop=True, stem=True)}
fields = {"название": 2.0, "формулировка": 1.0}
indexes = {name: Index(docs, p, fields, ids) for name, p in pipes.items()}
pd.DataFrame([{"конвейер": n, "шаги": p.describe(), "размер словаря": len(indexes[n].vocab),
               "средняя длина документа": round(indexes[n].avgdl, 1)} for n, p in pipes.items()])

              конвейер                                      шаги  размер словаря  средняя длина документа
0     без нормализации                                    токены             239                     13.7
1           стоп-слова                    токены → без стоп-слов             223                     11.5
2  стоп-слова и основы  токены → без стоп-слов → основы Snowball             173                     11.5


## Шаг 2. Инвертированный индекс

Индекс есть словарь «слово → документы, где оно встречается, с частотой». Матрица «слово × документ»
почти пуста, и индекс хранит только ненулевые клетки. Булев запрос «рецензия И срок» есть пересечение
двух списков.

In [4]:
idx = indexes["стоп-слова и основы"]
for w in ("рецензия", "решение", "срок", "статья"):
    stem = idx.pipe(w)[0]
    print(f"{w:10} → основа {stem!r:12} df = {idx.df[stem]:2}  документы: {[ids[i] for i in idx.docs_with(w)]}")
print()
print("«рецензия И срок»:", [ids[i] for i in idx.boolean_and("рецензия срок")])
nonzero = sum(len(p) for p in idx.postings.values())
print(f"матрица {len(idx.vocab)} слов × {idx.N} документов = {len(idx.vocab) * idx.N} клеток, ненулевых {nonzero} ({100 * nonzero / (len(idx.vocab) * idx.N):.1f} %)")

рецензия   → основа 'реценз'     df =  8  документы: ['T06', 'T08', 'S06', 'R61', 'R63', 'R64', 'R65', 'R66']
решение    → основа 'решен'      df =  5  документы: ['S01', 'T09', 'T10', 'R65', 'R91']
срок       → основа 'срок'       df =  1  документы: ['R66']
статья     → основа 'стат'       df = 18  документы: ['S01', 'T01', 'T02', 'T03', 'T04', 'T05', 'T06', 'T07', 'T09', 'S05', 'R51', 'R53', 'R61', 'R62', 'R63', 'R82', 'R83', 'R91']

«рецензия И срок»: ['R66']
матрица 173 слов × 31 документов = 5363 клеток, ненулевых 305 (5.7 %)


## Шаг 3. TF-IDF и косинус

Вес слова в документе есть частота, умноженная на редкость: TF · log(N / df). Запрос тоже вектор,
близость есть косинус угла. Вопрос 2 редактора из лекции 1.

In [5]:
q2 = "На какие положения опирается требование 6.5 о принятии решения"
print("запрос после конвейера:", idx.pipe(q2))
print("IDF слов запроса:", {w: round(idx.idf_plain(w), 2) for w in idx.pipe(q2)})
print()
for d, s in idx.tfidf(q2, k=5):
    print(f"{s:6.3f}  {d:4}  {text[d][:80]}")

запрос после конвейера: ['как', 'положен', 'опира', 'требован', '6.5', 'принят', 'решен']
IDF слов запроса: {'как': 0.0, 'положен': 2.34, 'опира': 0.0, 'требован': 2.34, '6.5': 3.43, 'принят': 2.74, 'решен': 1.82}

 0.482  R65   6.5 Принятие решения. Редакционная коллегия принимает решение на основании не ме
 0.395  S01   Раздел 1. Область применения. Положение устанавливает требования к подготовке ре
 0.193  S05   Раздел 5. Требования к оформлению статьи. Раздел устанавливает требования к объё
 0.111  T10   Решение. Решение — заключение редакционной коллегии: принять принять после дораб
 0.100  S08   Раздел 8. Этика и ответственность. Раздел устанавливает требования к раскрытию к


## Шаг 4. BM25

Две поправки к TF-IDF: насыщение частоты (k1) и нормализация длины (b). Оценка документа есть сумма по словам
запроса: IDF · f (k1 + 1) / (f + k1 (1 − b + b · dl / avgdl)). Та же выдача, и порядок другой:
раздел 1 поднялся выше пункта 6.5.

In [6]:
for d, s in idx.bm25(q2, k=5):
    print(f"{s:6.3f}  {d:4}  {text[d][:80]}")
print()
for d in ("S01", "R65"):
    i = ids.index(d)
    parts = idx.bm25_score(q2, i, explain=True)
    print(f"{d}: длина {idx.dl[i]:2}, вклад слов {dict((w, round(v, 2)) for w, v in parts.items())}")
print()
print("У S01 совпали четыре основы, у R65 две: «принятии» в запросе и «принимает» в 6.5 дают разные основы.")

 7.952  S01   Раздел 1. Область применения. Положение устанавливает требования к подготовке ре
 7.456  R65   6.5 Принятие решения. Редакционная коллегия принимает решение на основании не ме
 3.116  S05   Раздел 5. Требования к оформлению статьи. Раздел устанавливает требования к объё
 2.625  T10   Решение. Решение — заключение редакционной коллегии: принять принять после дораб
 2.256  S03   Раздел 3. Термины и определения. В положении применяются термины с соответствующ

S01: длина 14, вклад слов {'положен': 2.01, 'требован': 2.01, 'принят': 2.32, 'решен': 1.6}
R65: длина 14, вклад слов {'6.5': 2.79, 'принят': 2.32, 'решен': 2.35}

У S01 совпали четыре основы, у R65 две: «принятии» в запросе и «принимает» в 6.5 дают разные основы.


**Параметры.** k1 задаёт, как быстро насыщается частота: при k1 → 0 повторы слова не считаются, при большом k1
вес растёт почти линейно. b задаёт, насколько учитывать длину документа: b = 0 длина не учитывается, b = 1 учитывается полностью.
На коротком корпусе порядок выдачи от параметров почти не зависит, оценки меняются.

In [7]:
q = "требования к статье"
table = []
for k1 in (0.5, 1.5, 3.0):
    for b in (0.0, 0.75, 1.0):
        top = idx.bm25(q, k=3, k1=k1, b=b)
        table.append({"k1": k1, "b": b, "выдача": "  ".join(f"{d} {s}" for d, s in top)})
pd.DataFrame(table)

    k1     b                           выдача
0  0.5  0.00  S05 3.313  S01 2.761  S08 2.213
1  0.5  0.75  S05 3.291  S01 2.618  S08 2.142
2  0.5  1.00   S05 3.284  S01 2.573  S08 2.12
3  1.5  0.00  S05 3.944  S01 2.761  S08 2.213
4  1.5  0.75  S05 3.888  S01 2.513  S08 2.089
5  1.5  1.00    S05 3.87  S01 2.44  S08 2.051
6  3.0  0.00  S05 4.418  S01 2.761  S08 2.213
7  3.0  0.75    S05 4.33  S01 2.458  S08 2.06
8  3.0  1.00  S05 4.302  S01 2.371  S08 2.014


**Мини-практика: разбор ранжирования вручную.** Три документа, запрос из двух слов. Таблица частот и длин,
затем оценка по формуле для каждого документа, затем сверка с функцией.

In [8]:
toy = [{"т": "Рецензия содержит оценку и замечания."},
       {"т": "Замечание рецензии указывает на фрагмент статьи."},
       {"т": "Статья содержит аннотацию, ключевые слова и список литературы; рецензия на статью назначается редколлегией."}]
ti = Index(toy, Pipeline(), {"т": 1.0}, ["d1", "d2", "d3"])
qt = "замечание рецензии"
k1, b = 1.5, 0.75
print("основы запроса:", ti.pipe(qt), "  N =", ti.N, " avgdl =", round(ti.avgdl, 2))
for w in ti.pipe(qt):
    print(f"  {w!r}: df = {ti.df.get(w, 0)}, IDF = log((N - df + 0.5)/(df + 0.5) + 1) = {ti.idf_bm25(w):.3f}")
print()
for i, d in enumerate(ti.ids):
    tf = collections.Counter(ti.tokens[i]); dl = ti.dl[i]
    norm = 1 - b + b * dl / ti.avgdl
    terms = []
    total = 0.0
    for w in ti.pipe(qt):
        f = tf.get(w, 0)
        if f:
            v = ti.idf_bm25(w) * f * (k1 + 1) / (f + k1 * norm)
            terms.append(f"{ti.idf_bm25(w):.3f}·{f}·{k1 + 1}/({f} + {k1}·{norm:.2f}) = {v:.3f}")
            total += v
    print(f"{d}: dl = {dl}, норма длины = {norm:.2f}")
    for t in terms: print("    ", t)
    print(f"     оценка = {total:.3f}   функция даёт {ti.bm25_score(qt, i, k1, b):.3f}")
print("\nвыдача:", ti.bm25(qt, k=3))

основы запроса: ['замечан', 'реценз']   N = 3  avgdl = 6.67
  'замечан': df = 2, IDF = log((N - df + 0.5)/(df + 0.5) + 1) = 0.470
  'реценз': df = 3, IDF = log((N - df + 0.5)/(df + 0.5) + 1) = 0.134

d1: dl = 4, норма длины = 0.70
     0.470·1·2.5/(1 + 1.5·0.70) = 0.573
     0.134·1·2.5/(1 + 1.5·0.70) = 0.163
     оценка = 0.736   функция даёт 0.736
d2: dl = 5, норма длины = 0.81
     0.470·1·2.5/(1 + 1.5·0.81) = 0.530
     0.134·1·2.5/(1 + 1.5·0.81) = 0.150
     оценка = 0.680   функция даёт 0.680
d3: dl = 11, норма длины = 1.49
     0.134·1·2.5/(1 + 1.5·1.49) = 0.103
     оценка = 0.103   функция даёт 0.103

выдача: [('d1', 0.736), ('d2', 0.68), ('d3', 0.103)]


## Шаг 5. BM25F: поля документа

У записи положения два поля. Название короткое и точное, формулировка длиннее. BM25F складывает частоты по полям
с весами, у названия вес 2, и нормирует длину каждого поля отдельно. Запрос «срок рецензирования» совпадает
с названием пункта 6.6 целиком.

In [9]:
for q in ("срок рецензирования", q2):
    print(q)
    print("   BM25 :", idx.bm25(q, k=4))
    print("   BM25F:", idx.bm25f(q, k=4))

срок рецензирования
   BM25 : [('R66', 5.119), ('R81', 2.083), ('S01', 1.786), ('S06', 1.786)]
   BM25F: [('R66', 6.878), ('S06', 2.406), ('R81', 2.164), ('S01', 1.852)]
На какие положения опирается требование 6.5 о принятии решения
   BM25 : [('S01', 7.952), ('R65', 7.456), ('S05', 3.116), ('T10', 2.625)]
   BM25F: [('R65', 10.498), ('S01', 8.249), ('S05', 3.396), ('T10', 3.343)]


## Шаг 6. Тестовая выборка

Шестнадцать запросов к положению. Первые пять суть вопросы редактора из лекции 1, остальные задаёт положению
редактор или автор в работе. Для каждого запроса размечены релевантные документы с градацией: 2 отвечает на вопрос,
1 нужен для ответа или уточняет, 0 всё остальное. Разметил автор корпуса, который знает ответы.

In [10]:
qrels = load_qrels("data/queries_Aprime.csv")
pd.DataFrame([{"id": i, "запрос": q, "релевантные": " ".join(f"{d}:{g}" for d, g in rel.items())} for i, q, rel in qrels])

     id                                                          запрос                    релевантные
0   q01                       Какие статьи готовы к решению редколлегии  R65:2 R61:1 R64:1 R66:1 T10:1
1   q02  На какие положения опирается требование 6.5 о принятии решения  R65:2 R61:2 R64:2 T10:1 S06:1
2   q03      Может ли рецензент состоять в одной организации с авторами                    R62:2 R81:1
3   q04                             Кто назначает рецензентов на статью        T05:2 S06:1 T09:1 R61:1
4   q05                      Сколько недель даётся на подготовку отзыва                          R66:2
5   q06                     Сколько рецензий нужно для принятия решения                    R61:2 R65:2
6   q07                                        Объём статьи в страницах                    R51:2 T01:1
7   q08                                          Требования к аннотации                    R52:2 T02:1
8   q09                                  Что такое замечание рецензента  

## Шаг 7. Метрики на выборке

Точность@5: доля релевантных среди первых пяти. Полнота@5: доля найденных среди релевантных. MRR: средний обратный
ранг первого релевантного. MAP: средняя точность по позициям релевантных. NDCG@10: выигрыш со скидкой по позиции,
делённый на выигрыш идеального порядка. Четыре варианта: три ранжирования на полном конвейере и BM25 без нормализации.

In [11]:
variants = {
    "TF-IDF, косинус": lambda q: idx.tfidf(q, k=10),
    "BM25": lambda q: idx.bm25(q, k=10),
    "BM25F, название ×2": lambda q: idx.bm25f(q, k=10),
    "BM25 без стоп-слов и основ": lambda q: indexes["без нормализации"].bm25(q, k=10),
}
metrics = pd.DataFrame({name: evaluate(fn, qrels) for name, fn in variants.items()}).T
metrics

                              P@5    R@5    MRR    MAP  NDCG@10
TF-IDF, косинус             0.338  0.665  0.938  0.631    0.768
BM25                        0.338  0.665  0.865  0.617    0.735
BM25F, название ×2          0.338  0.665  0.906  0.633    0.753
BM25 без стоп-слов и основ  0.250  0.512  0.762  0.478    0.619


In [12]:
# тот же BM25F по каждому запросу: где поиск справляется и где нет
per_q = []
for i, q, rel in qrels:
    ranked = [d for d, _ in idx.bm25f(q, k=10)]
    per_q.append({"id": i, "P@5": round(precision_at_k(ranked, rel, 5), 2), "RR": round(reciprocal_rank(ranked, rel), 2),
                  "NDCG@10": round(ndcg_at_k(ranked, rel, 10), 2), "первые 5": " ".join(ranked[:5])})
pd.DataFrame(per_q)

     id  P@5   RR  NDCG@10             первые 5
0   q01  0.4  1.0     0.57  T10 R65 T09 S01 R91
1   q02  0.4  1.0     0.48  R65 S01 S05 T10 S03
2   q03  0.4  1.0     1.00  R62 R81 T05 T04 S06
3   q04  0.4  1.0     0.72  R61 T05 R62 T06 T07
4   q05  0.0  0.0     0.00          R82 S01 T11
5   q06  0.2  1.0     0.82  R65 S01 T10 T06 R63
6   q07  0.4  1.0     0.95  R51 S05 T01 R82 T03
7   q08  0.4  1.0     0.86  T02 R52 S05 S08 S01
8   q09  0.4  1.0     0.90  T07 T06 R64 T05 R62
9   q10  0.4  1.0     0.95  T08 R64 T06 R63 T07
10  q11  0.6  1.0     0.96  R82 T11 S08 S01 R83
11  q12  0.4  1.0     1.00  R83 T04 R63 S08 T02
12  q13  0.2  1.0     0.76  R91 T01 R51 S05 R82
13  q14  0.4  1.0     0.92  R81 S08 T05 R62 R61
14  q15  0.2  0.5     0.39  T10 R65 T09 S01 R91
15  q16  0.2  1.0     0.76          R54 R51 R52


## Шаг 8. NDCG@3 вручную

Выдача BM25F по вопросу 2. Выигрыш позиции есть градация документа, скидка есть log₂(позиция + 1).
DCG@3 суммирует три первых выигрыша со скидкой, IDCG@3 считает то же для идеального порядка градаций, NDCG есть отношение.

In [13]:
_, q, rel = qrels[1]
ranked = [d for d, _ in idx.bm25f(q, k=3)]
gains = [rel.get(d, 0) for d in ranked]
ideal = sorted(rel.values(), reverse=True)[:3]
print("запрос:", q)
print("выдача:", ranked, " градации:", gains, " идеальные градации:", ideal)
parts = [f"{g}/log2({p + 1}) = {g / math.log2(p + 1):.3f}" for p, g in enumerate(gains, 1)]
print("DCG@3  =", " + ".join(parts), "=", round(dcg(gains, 3), 3))
parts = [f"{g}/log2({p + 1}) = {g / math.log2(p + 1):.3f}" for p, g in enumerate(ideal, 1)]
print("IDCG@3 =", " + ".join(parts), "=", round(dcg(ideal, 3), 3))
print("NDCG@3 =", round(dcg(gains, 3) / dcg(ideal, 3), 3), "  функция:", round(ndcg_at_k(ranked, rel, 3), 3))

запрос: На какие положения опирается требование 6.5 о принятии решения
выдача: ['R65', 'S01', 'S05']  градации: [2, 0, 0]  идеальные градации: [2, 2, 2]
DCG@3  = 2/log2(2) = 2.000 + 0/log2(3) = 0.000 + 0/log2(4) = 0.000 = 2.0
IDCG@3 = 2/log2(2) = 2.000 + 2/log2(3) = 1.262 + 2/log2(4) = 1.000 = 4.262
NDCG@3 = 0.469   функция: 0.469


## Шаг 9. Где поиск по словам ломается

Три запроса, на которых ни один параметр BM25 не помогает. Синонимия: слова запроса и документа разные
при одном смысле. Перефразировка: тот же вопрос другими словами. Формула: запись символами против записи словами.

In [14]:
def show(index, q, rel, k=5):
    out = index.bm25f(q, k=k)
    found = [d for d, _ in out]
    print(f"запрос: {q}\n  основы: {index.pipe(q)}\n  выдача: {out if out else 'пусто'}")
    for d, g in rel.items():
        pos = found.index(d) + 1 if d in found else None
        print(f"  релевантный {d} (градация {g}): {'позиция ' + str(pos) if pos else 'не найден'}")

print("1. Синонимия")
show(idx, qrels[4][1], qrels[4][2])
print("   6.6 написан словами «рецензия», «21 дня», «назначения». Запрос говорит «отзыв», «недель», «подготовку». Общих основ нет.\n")
print("2. Перефразировка")
show(idx, qrels[14][1], qrels[14][2])
print("   6.1 говорит «не менее двух рецензентов», запрос говорит «мнений экспертов». Найдено по словам «решение» и «статья», 6.1 не найден.\n")
idxA = Index(docsA, Pipeline(), fields, idsA)
qrelsA = load_qrels("data/queries_A.csv")
print("3. Формула")
show(idxA, qrelsA[1][1], qrelsA[1][2])
print("   Запись a² + b² = c² распадается на буквы a, b, c, и они есть в формуле Герона. Теорема Пифагора записана словами.\n")
print("4. Связи (для лекции 7)")
show(idxA, qrelsA[2][1], qrelsA[2][2])
print("   Поиск находит саму теорему. Лемму о высоте и определения, от которых она зависит, по словам не найти: нужен граф.")

1. Синонимия
запрос: Сколько недель даётся на подготовку отзыва
  основы: ['скольк', 'недел', 'дает', 'подготовк', 'отзыв']
  выдача: [('R82', 2.195), ('S01', 2.089), ('T11', 1.826)]
  релевантный R66 (градация 2): не найден
   6.6 написан словами «рецензия», «21 дня», «назначения». Запрос говорит «отзыв», «недель», «подготовку». Общих основ нет.

2. Перефразировка
запрос: Сколько мнений экспертов нужно для решения по статье
  основы: ['скольк', 'мнен', 'эксперт', 'нужн', 'решен', 'стат']
  выдача: [('T10', 3.343), ('R65', 2.816), ('T09', 2.547), ('S01', 2.18), ('R91', 2.08)]
  релевантный R61 (градация 2): не найден
  релевантный R65 (градация 2): позиция 2
   6.1 говорит «не менее двух рецензентов», запрос говорит «мнений экспертов». Найдено по словам «решение» и «статья», 6.1 не найден.

3. Формула
запрос: a² + b² = c²
  основы: ['a', 'b', 'c']
  выдача: [('A29', 11.015)]
  релевантный A14 (градация 2): не найден
  релевантный A15 (градация 1): не найден
   Запись a² + b² = c² распа

## Шаг 10. Агент: инструмент поиска с измеренным качеством

Инструмент `search_words` из лекции 1 получает в описании измеренную точность на выборке. Инструкция требует
помечать найденное «найдено по словам» с рангом, а пустую выдачу сообщать как «не найдено по словам».
Прогон без модели, по записи: инструменты вызываются по-настоящему.

In [15]:
from agent_core import run_agent, scripted_llm

P5 = metrics.loc["BM25F, название ×2", "P@5"]

def search_words(query: str, k: int = 5):
    return [{"id": d, "оценка": s, "текст": text[d]} for d, s in idx.bm25f(query, k=k)]

TOOLS = [{"type": "function", "function": {
    "name": "search_words",
    "description": f"Поиск по словам (BM25F) по положению о рецензировании. Точность@5 на выборке из 16 запросов: {P5}. "
                   "Находит документы, где есть слова запроса. Синонимов и связей между документами не знает.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string"}, "k": {"type": "integer"}}, "required": ["query"]}}}]
IMPL = {"search_words": search_words}
SYSTEM = ("Ты помощник редактора. Сначала вызови search_words. Каждый документ в ответе помечай [найдено по словам: ранг]. "
          "Если выдача пуста или релевантного документа в ней нет, напиши [не найдено по словам] и не выдумывай.")

plan = [("search_words", {"query": q2, "k": 5})]
answer = ("Требование 6.5 (R65): редколлегия принимает решение на основании не менее двух рецензий и ответа авторов "
          "[найдено по словам: ранг 1]. Раздел 1 (S01) задаёт область применения [найдено по словам: ранг 2]. "
          "Какие положения стоят за 6.5 (6.1, 6.4, термин «Решение»), по словам не найти: нужен граф зависимостей.")
_ = run_agent(q2, scripted_llm(plan, answer), SYSTEM, TOOLS, IMPL)

[1] модель вызвала search_words({'query': 'На какие положения опирается требование 6.5 о принятии решения', 'k': 5})
    <- [{"id": "R65", "оценка": 10.498, "текст": "6.5 Принятие решения. Редакционная коллегия принимает решение на основании не менее двух рецензий и ответа авторов."}…
[2] ответ модели:
Требование 6.5 (R65): редколлегия принимает решение на основании не менее двух рецензий и ответа авторов [найдено по словам: ранг 1]. Раздел 1 (S01) задаёт область применения [найдено по словам: ранг 2]. Какие положения стоят за 6.5 (6.1, 6.4, термин «Решение»), по словам не найти: нужен граф зависимостей.


In [16]:
q5 = qrels[4][1]
plan = [("search_words", {"query": q5, "k": 5})]
answer = ("[не найдено по словам] Выдача содержит 8.2, раздел 1 и термин «ИИ-инструмент», совпавшие по слову «подготовка». "
          "Нормы о сроке среди них нет. Вопрос нужно задать иначе или другим каналом.")
_ = run_agent(q5, scripted_llm(plan, answer), SYSTEM, TOOLS, IMPL)

[1] модель вызвала search_words({'query': 'Сколько недель даётся на подготовку отзыва', 'k': 5})
    <- [{"id": "R82", "оценка": 2.195, "текст": "8.2 Использование ИИ-инструментов. Использование ИИ-инструментов при подготовке статьи указывается в отдельном разделе…
[2] ответ модели:
[не найдено по словам] Выдача содержит 8.2, раздел 1 и термин «ИИ-инструмент», совпавшие по слову «подготовка». Нормы о сроке среди них нет. Вопрос нужно задать иначе или другим каналом.


## Шаг 11. Вариант по математике

Тот же конвейер и те же метрики на корпусе планиметрии и выборке из десяти вопросов исследователя.

In [17]:
pd.DataFrame({"BM25": evaluate(lambda q: idxA.bm25(q, k=10), qrelsA),
              "BM25F": evaluate(lambda q: idxA.bm25f(q, k=10), qrelsA)}).T

        P@5    R@5  MRR    MAP  NDCG@10
BM25   0.38  0.742  0.9  0.708    0.796
BM25F  0.38  0.742  0.9  0.708    0.799


## Итог

Индекс отвечает на вопрос «где есть эти слова». BM25 упорядочивает ответ по редкости и частоте слов с поправкой на длину.
Метрики на размеченной выборке показывают, насколько порядок хорош, и та же выборка послужит для векторного (лекция 6),
графового (лекция 7) и гибридного (лекция 8) поиска. Три провала, синонимия, перефразировка и формулы, не лечатся параметрами:
это граница канала.